# Model Selection

In [1]:
import numpy as np
import pandas as pd
import joblib
from src.config import GENE_COLS, BMI_AGE_COLS, TARGET_COL

In [30]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
from src.evaluate import scoring, specificity_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

Import test and train indices, preprocessor, and stable features selected in previous notebook

Import dataset and shave down to stable features, then replicate train-test-split

In [6]:
stable_features = joblib.load('../models/stable_features.pk1')
df = pd.read_csv('../data/processed/clean_data.csv', usecols = stable_features + [TARGET_COL])
train_idx = np.load('../data/processed/train_idx.npy', allow_pickle=True)
test_idx = np.load('../data/processed/test_idx.npy', allow_pickle=True)
preprocessor = joblib.load('../models/preprocessor.pk1')
df.head()

,BMI,age,has_diabetes,mtb_0522146,mtb_0963388,mtb_0969469,mtb_0985311,mtb_1221553,mtb_1311918,mtb_1374393,mtb_1425338,mtb_1567645,mtb_1766602,mtb_1827483,mtb_1897069
0,18.664268,33.81,0.0,298469.0,1261009.0,738104.8,4268873.0,1125632.0,538720.6,1382281.0,3579584.0,2076190.0,2437778.0,20943080.0,443472.0
1,28.175977,68.56,0.0,291403.9,463440.1,657937.1,1646335.0,1024647.0,203498.7,648712.1,2225307.0,878822.9,1643203.0,2540190.0,163333.8
2,22.971959,55.68,0.0,237327.5,1527346.0,495850.8,3814640.0,3745193.0,1501483.0,1675639.0,4457760.0,1769644.0,1961078.0,15926480.0,577703.9
3,21.960370,43.89,0.0,218136.1,1925192.0,525130.6,4106548.0,1596852.0,2039977.0,1359338.0,3467916.0,1484457.0,2228357.0,8809436.0,318221.4
4,40.454949,47.76,0.0,306971.8,2704172.0,650922.4,2253769.0,791436.3,549498.9,1050007.0,3480523.0,1281824.0,1702022.0,7623808.0,301541.4


In [7]:
X_train, X_test = df.loc[train_idx].drop(columns=TARGET_COL), df.loc[test_idx].drop(columns=TARGET_COL)
y_train, y_test = df.loc[train_idx, TARGET_COL], df.loc[test_idx, TARGET_COL]

## Define prioritization of evaluation metrics  
1. Recall (sensitivity) - catch all positive cases, avoiding missed detections (false negatives)
2. Specificity - correctly identify negative cases, avoiding false alarms

Recall will be our highest priority metric for this blood screen classifier, as we do not want to miss any true positive cases. Next most important is specificity, since we would want to avoid false alarms.

Since we have a class imbalance, AUC-PR will be more informative than AUC-ROC to assess model performance, and to compare between models.

### Benchmark - dummy classifier
Anything that cannot beat this isn't worth deploying. This baseline will be used as a sanity check to ensure that any model we test at least beats random chance.

In [26]:
X_train_transformed = preprocessor.transform(X_train)
dummy = DummyClassifier(strategy='stratified')
dummy.fit(X_train_transformed, y_train)
print(classification_report(y_test, dummy.predict(preprocessor.transform(X_test)), zero_division=0))

              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85      1399
         1.0       0.12      0.13      0.12       231

    accuracy                           0.75      1630
   macro avg       0.49      0.49      0.49      1630
weighted avg       0.75      0.75      0.75      1630



## Start exploring more interpretable models  
1. Logistic regression
2. Linear SVM
3. Decision tree

In [28]:
models = {
    'logistic_regression' : LogisticRegression(class_weight='balanced', max_iter=1000),
    'linear_svm' : SVC(kernel='linear', class_weight='balanced', probability=True),
    'decision_tree' : DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=33)
}

Perform cross validation and obtain relevant metrics across each cv fold:

In [32]:
cv_results = {}
for name, model in models.items():

    # first, connect the preprocessor with each model to test out
    pipeline = Pipeline(
        steps = [
            ('preprocess', preprocessor),
            ('model', model)
        ]
    )

    # second, perform cross validation and obtain metrics consistent with our scoring table
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # third, store the cross-validated results, and compare across folds using std
    cv_results[name] = {
        'auc_pr_mean' : scores['test_auc_pr'].mean(),
        'auc_pr_std' : scores['test_auc_pr'].std(),
        'auc_roc_mean' : scores['test_auc_roc'].mean(),
        'auc_roc_std' : scores['test_auc_roc'].std(),
        'recall_mean' : scores['test_recall'].mean(), # sensitivity - our highest priority metric
        'recall_std' : scores['test_recall'].std(),
        'specificity_mean' : scores['test_specificity'].mean(),
        'specificity_std' : scores['test_specificity'].std(),
    }

In [34]:
results_df = pd.DataFrame(cv_results).T
results_df.round(4)

,auc_pr_mean,auc_pr_std,auc_roc_mean,auc_roc_std,recall_mean,recall_std,specificity_mean,specificity_std
logistic_regression,0.4162,0.0375,0.8045,0.0185,0.7456,0.0337,0.7217,0.0168
linear_svm,0.4135,0.0371,0.8041,0.0184,0.7597,0.0391,0.7058,0.0170
decision_tree,0.3335,0.0299,0.7494,0.0224,0.7165,0.0608,0.6750,0.0290


The logistic regression and linear SVM models performed similarly, while the decision tree is worse.

Looking at the results of logistic regression and our SVM, the AUC-PR is quite low at 0.4, while the recall is much higher at around 0.75. This indicates that the model is achieving higher recall at the expense of low precision. These models are catching most diabetics, but is greedily flagging a large portion of nondiabetics as well.